In [1]:
import pandas as pd
import numpy
import scipy.stats as stats

import numpy as np

def calculate_cpm(count_matrix):
    """
    Calculate Counts Per Million (CPM) for each column (sample) in a count matrix.

    Parameters
    ----------
    count_matrix : numpy.ndarray or pandas.DataFrame
        2D array where rows are features (genes) and columns are samples.
        Can be a NumPy array or a pandas DataFrame.

    Returns
    -------
    cpm_matrix : numpy.ndarray
        CPM-normalized values in the same shape as input.
    """
    # If input is DataFrame, convert to numpy array
    if hasattr(count_matrix, 'values'):
        counts = count_matrix.values
    else:
        counts = np.array(count_matrix)

    # Sum counts per sample (column sum)
    col_sums = counts.sum(axis=0)

    # Avoid division by zero (replace zeros with 1 to prevent inf, or handle as needed)
    col_sums[col_sums == 0] = 1

    # Calculate CPM
    cpm_matrix = (counts / col_sums) * 1e6

    return cpm_matrix


In [2]:
df = pd.read_csv("brainMetPairs.salmon.cts.txt", sep = "\t", index_col=0)

df_cpm = calculate_cpm(df)
df_cpm = np.log(df_cpm + 1)
df_cpm = pd.DataFrame(df_cpm, index=df.index, columns=df.columns)

#parsing sample type column
meta_data = pd.DataFrame({"sample_id" : df.columns, "condition" : df.columns}, index=df.columns)
meta_data["condition"] = [list(x.split("_")[0])[-1] for x in meta_data.condition]
meta_data = meta_data.loc[meta_data.condition == "M"]

# from reference: samples ‘7M_RCS’ and ‘19.2M_Pitt’ were dropped, the first due to a lack of matching clinical data and the second due to sample replication
sample_of_interest = [x for x in meta_data.sample_id if x not in ["7M_RCS", "19-2M_Pitt"]]
meta_data_subset = meta_data.loc[meta_data.sample_id.isin(sample_of_interest)]

clinical_data = pd.read_csv("clinical_data_for_survival.txt", sep = "\t")
clinical_data = clinical_data.loc[clinical_data.sample_id != "19_2_Pitt"]
clinical_data.index = list(clinical_data.sample_id)


FileNotFoundError: [Errno 2] No such file or directory: 'brainMetPairs.salmon.cts.txt'

In [ ]:
#gene_symbols = ["ENSG00000100146", "ENSG00000205927", "ENSG00000184221", "ENSG00000123560", "ENSG00000105695", "ENSG00000204655", "ENSG00000197430", "ENSG00000196136", "ENSG00000224389"]
gene_symbols = ["ENSG00000224389"]

df_subset = df_cpm.loc[df_cpm.index.isin(gene_symbols)]

## subset of gene expression for sample
df_subset = df_subset[meta_data_subset.sample_id]

meta_data_subset["sample_id"] = [x.replace("M", "") for x in meta_data_subset.sample_id]
meta_data_subset.index = list(meta_data_subset.sample_id)
surv_data = pd.merge(meta_data_subset, clinical_data, left_index=True, right_index=True)

if(len(gene_symbols) > 1):
  # Reference: the sum of the scaled data (z-scores) for multigene signatures. NOTE: this is for the gene signature and not for the individual gene.
  df_subset_z_score = stats.zscore(                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       )
  df_subset_z_score = pd.DataFrame(df_subset_z_score, index=df_subset.index, columns=df_subset.columns)
  surv_data["score"] = np.array(df_subset_z_score.sum())
else:
  surv_data["score"] = np.array(df_subset.sum())

median_score = np.median(surv_data.score)

surv_data["group"] = np.array(["high" if x > median_score else "low" for x in surv_data.score])
surv_data = surv_data[['sample_id_x', 'condition', 'sample_id_y', 'Status', 'DFS', 'BMSF', 'SPBM', 'OS', 'group']]

surv_data.to_csv("surv_data_oligos C4b.csv")

/tmp/ipython-input-61-2792989034.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_data_subset["sample_id"] = [x.replace("M", "") for x in meta_data_subset.sample_id]
